# Pauli Propagation

In [2]:
import math
from functools import partial
import numpy as np
from matplotlib import pyplot as plt

from qiskit.quantum_info import Statevector, Operator
from quantum_simulation_recipe.plot_config import *
from quantum_simulation_recipe.trotter import pf, expH
from quantum_simulation_recipe.spin import Nearest_Neighbour_1d

from trotter_bounds import *
from pauli import *

fig_dir, data_dir = './figs', './data'
# plt.rcParams['font.family'] = 'sans-serif'
set_fontsize(linewidth=2.0)

In [3]:
from qiskit.quantum_info import SparsePauliOp
import sympy as sp
from collections import defaultdict

class SymbolicSparsePauliOp:
    def __init__(self, pauli_strings, symbolic_coeffs):
        """
        pauli_strings: list of Pauli strings
        symbolic_coeffs: list of sympy expressions or numbers
        """
        self.terms = {}
        for ps, coeff in zip(pauli_strings, symbolic_coeffs):
            # Ensure coefficient is a sympy expression
            if not isinstance(coeff, sp.Basic):
                coeff = sp.sympify(coeff)
            
            if ps in self.terms:
                self.terms[ps] = self.terms[ps] + coeff
            else:
                self.terms[ps] = coeff
    
    def __add__(self, other):
        """Add two SymbolicSparsePauliOp objects"""
        result_terms = self.terms.copy()
        for ps, coeff in other.terms.items():
            if ps in result_terms:
                result_terms[ps] = result_terms[ps] + coeff
            else:
                result_terms[ps] = coeff
        
        # Create new object with combined terms
        pauli_strings = list(result_terms.keys())
        coeffs = list(result_terms.values())
        return SymbolicSparsePauliOp(pauli_strings, coeffs)
    
    def __mul__(self, scalar):
        """Multiply by a scalar (symbolic or numeric)"""
        if not isinstance(scalar, sp.Basic):
            scalar = sp.sympify(scalar)
        new_coeffs = [scalar * coeff for coeff in self.terms.values()]
        return SymbolicSparsePauliOp(list(self.terms.keys()), new_coeffs)
    
    def __rmul__(self, scalar):
        return self.__mul__(scalar)
    
    def evaluate(self, **values):
        """Evaluate with specific values for symbols"""
        numeric_coeffs = []
        pauli_strings = []
        
        # Convert string keys to Symbol objects for substitution
        subs_dict = {}
        for key, value in values.items():
            # Create symbol with the same name and assumptions
            sym = sp.Symbol(key)
            subs_dict[sym] = value
        
        for ps, coeff in self.terms.items():
            # Get all symbols in this coefficient
            symbols_in_coeff = coeff.free_symbols
            
            # Build substitution dictionary only for symbols present in this coefficient
            local_subs = {}
            for sym in symbols_in_coeff:
                if sym.name in values:
                    local_subs[sym] = values[sym.name]
            
            # Substitute
            substituted = coeff.subs(local_subs)
            
            # Check if all symbols were substituted
            if substituted.has(sp.Symbol):
                free_symbols = substituted.free_symbols
                missing = [s.name for s in free_symbols]
                raise ValueError(f"Not all symbols were given values. Missing: {missing}")
            
            # Convert to complex number
            try:
                # Handle both real and complex expressions
                numeric_value = complex(substituted.evalf())
            except (TypeError, AttributeError):
                # If it's already a number
                numeric_value = complex(substituted)
            
            # Only add non-zero terms
            if abs(numeric_value) > 1e-10:  # Tolerance for numerical zeros
                numeric_coeffs.append(numeric_value)
                pauli_strings.append(ps)
        
        if not pauli_strings:
            # If all coefficients are zero, return zero operator
            return SparsePauliOp(['I'], [0.0])
        
        return SparsePauliOp(pauli_strings, coeffs=numeric_coeffs)
    
    def __repr__(self):
        terms_str = []
        for ps, coeff in self.terms.items():
            terms_str.append(f"{coeff} * {ps}")
        return " + ".join(terms_str)


In [5]:
# Usage example
c, s = sp.symbols('c s', real=True)

# Create symbolic operators
op1 = SymbolicSparsePauliOp(['ZZ'], [c])
op2 = SymbolicSparsePauliOp(['XX'], [s])

# Add them symbolically
combined_op = op1 + op2
print("Symbolic expression:", combined_op)

# Evaluate with specific values
result = combined_op.evaluate(c=0.5, s=0.3)
print("\nEvaluated with c=0.5, s=0.3:")
print(result)
combined_op2 = combined_op  * 2
combined_op2.evaluate(c=0.5, s=1.3)

Symbolic expression: c * ZZ + s * XX

Evaluated with c=0.5, s=0.3:
SparsePauliOp(['ZZ', 'XX'],
              coeffs=[0.5+0.j, 0.3+0.j])


SparsePauliOp(['ZZ', 'XX'],
              coeffs=[1. +0.j, 2.6+0.j])

In [8]:
c = 1
SparsePauliOp(['IIIIIIXX'], coeffs=[c])
# SparsePauliOp(['IIIIIIXX'], coeffs=[1])

SparsePauliOp(['IIIIIIXX'],
              coeffs=[1.+0.j])